In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
import scipy.stats as stats
import seaborn as sns
from sklearn.utils import resample


from sklearn.model_selection import train_test_split

In [ ]:
!pip install vaderSentiment


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import preprocessing
import xgboost as xgb

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

In [ ]:
import shap

Understanding the Dataset - Complaints


In [ ]:
complaint_df = pd.read_csv('databusiness50k_newdate_df.csv')

In [ ]:
complaint_df

In [ ]:
complaint_df.info()

In [ ]:
complaint_df = complaint_df[[ 'date_received','consumer_complaint', 'product', 'sub_product', 'issue',  'sub_issue', 'company', 'submitted_via','state', 'company_response','timely_response','dispute_flag']]

In [ ]:
complaint_df

In [ ]:
CATEGORICAL_COLS = [
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "company",
    "state",
    "submitted_via",
    "company_response",
    "timely_response"
]

TEXT_COL = 'consumer_complaint'
TARGET_COL = 'dispute_flag'

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if pd.isna(text):
        return 0.0
    text = str(text).strip()
    if not text:
        return 0.0
    # compound score in [-1, 1]
    return analyzer.polarity_scores(text)['compound']

complaint_df['sentiment'] = complaint_df[TEXT_COL].apply(get_sentiment)

complaint_df['sentiment'].describe()

In [ ]:
NUMERIC_COLS = ['sentiment']

In [ ]:
complaint_df.columns.tolist()

In [ ]:
complaint_df = complaint_df.fillna("Unknown")

In [ ]:
complaint_df

Training

In [ ]:
complaint_df = complaint_df.sort_values('date_received').reset_index(drop=True)

train_size = int(len(complaint_df) * 0.8)

train_df = complaint_df.iloc[:train_size].copy()
test_df  = complaint_df.iloc[train_size:].copy()


In [ ]:
print(train_df['date_received'].min(), train_df['date_received'].max())
print(test_df['date_received'].min(), test_df['date_received'].max())


In [ ]:
print (train_df.shape, test_df.shape)

In [ ]:

TARGET_COL = "dispute_flag"

def balance_train(df, target_col=TARGET_COL, random_state=42):
    # Split by class
    df_major = df[df[target_col] == 0]   # majority (0)
    df_minor = df[df[target_col] == 1]   # minority (1)

    print("Before balancing:")
    print(df[target_col].value_counts(), "\n")

    # Upsample minority to match majority
    df_minor_up = resample(
        df_minor,
        replace=True,
        n_samples=len(df_major),
        random_state=random_state
    )

    # Combine and shuffle
    df_bal = pd.concat([df_major, df_minor_up], axis=0)
    df_bal = df_bal.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("After balancing:")
    print(df_bal[target_col].value_counts(), "\n")

    return df_bal


In [ ]:
train_df_balanced = balance_train(train_df, TARGET_COL)

# Use the balanced train only for fitting
X_train = train_df_balanced.drop(columns=[TARGET_COL])
y_train = train_df_balanced[TARGET_COL]

# Test set stays untouched (real-world distribution)
X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

In [ ]:
print(test_df[TARGET_COL].value_counts())


Preprocessor

In [ ]:
text_transformer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9,
    stop_words='english',
    sublinear_tf=True,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b'
)

cat_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
      ('text', text_transformer, TEXT_COL),
      ('cat', cat_transformer, CATEGORICAL_COLS)
    ]
)

preprocessor_sentiment = ColumnTransformer(
    transformers=[
      ('text', text_transformer, TEXT_COL),
      ('cat', cat_transformer, CATEGORICAL_COLS),
      ('num',  'passthrough',    NUMERIC_COLS)
    ]
)



Random Forest Classifier

In [ ]:
rf_clf = RandomForestClassifier (
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt"
)

randomforest_pipeline = Pipeline(steps = [
    ('preprocess', preprocessor),
    ('model', rf_clf)
])

randomforest_sentiment_pipeline = Pipeline(steps = [
    ('preprocess', preprocessor_sentiment),
    ('model', rf_clf)
])

In [ ]:
print("Training Random Forest...")
randomforest_pipeline.fit(X_train, y_train)
rf_pred = randomforest_pipeline.predict(X_test)
rf_proba = randomforest_pipeline.predict_proba(X_test)[:,1]
print("Training Random Forest Done!")

In [ ]:
print("\n=== Random Forest ===")
print(classification_report(y_test, rf_pred))
print("AUC:", roc_auc_score(y_test, rf_proba))

print("=== Random Forest Confusion Matrix ===")
print(confusion_matrix(y_test, rf_pred))


In [ ]:
print("Training Random Forest with Sentiment Analysis...")
randomforest_sentiment_pipeline.fit(X_train, y_train)
rf_sentiment_pred = randomforest_sentiment_pipeline.predict(X_test)
rf_sentiment_proba = randomforest_sentiment_pipeline.predict_proba(X_test)[:,1]
print("Training Random Forest with Sentiment Analysis Done!")

In [ ]:
print("\n=== Random Forest with Sentiment Analysis ===")
print(classification_report(y_test, rf_sentiment_pred))
print("AUC:", roc_auc_score(y_test, rf_sentiment_proba))

print("=== Random Forest with Sentiment Analysis Confusion Matrix ===")
print(confusion_matrix(y_test, rf_sentiment_pred))

Radom Forest (with Sentiment Analysis) Feature Importance

In [ ]:
preprocessor = randomforest_sentiment_pipeline.named_steps['preprocess']
rf_model = randomforest_sentiment_pipeline.named_steps['model']

text_features_all = preprocessor.named_transformers_['text'].get_feature_names_out()
n_text_features = len(text_features_all)

full_importances = rf_model.feature_importances_

text_importances_all = full_importances[:n_text_features]


In [ ]:
def is_redaction(token: str) -> bool:
    clean = token.replace(" ", "").lower()
    return len(clean) > 0 and set(clean) <= {"x"}

phrase_mask = np.array([" " in f for f in text_features_all])

phrase_features = text_features_all[phrase_mask]
phrase_importances = text_importances_all[phrase_mask]


clean_mask = np.array([not is_redaction(f) for f in phrase_features])
phrase_features_clean = phrase_features[clean_mask]
phrase_importances_clean = phrase_importances[clean_mask]


In [ ]:

top_k = 20
idx = np.argsort(phrase_importances_clean)[::-1][:top_k]

top_phrases = phrase_features_clean[idx]
top_scores  = phrase_importances_clean[idx]

print("Top 20 narrative PHRASES driving escalation:\n")
for i, (p, s) in enumerate(zip(top_phrases, top_scores), start=1):
    print(f"{i:2d}. {p}: {s:.5f}")

plt.figure(figsize=(8, 6))
plt.barh(range(top_k), top_scores[::-1])
plt.yticks(range(top_k), top_phrases[::-1])
plt.xlabel("Feature importance (text phrases only)")
plt.title("Top 20 narrative phrases driving escalation")
plt.tight_layout()
plt.show()


Logistic Regressiom

In [ ]:
logreg = LogisticRegression(
    max_iter=500,
    C=0.5,         # <--- stronger regularization
    solver="lbfgs",
    n_jobs=-1
)

logreg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", logreg)
])

logreg_sentiment_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", logreg)
])

In [ ]:
print("Training Logistic Regression...")
logreg_pipeline.fit(X_train, y_train)
logreg_pred = logreg_pipeline.predict(X_test)
logreg_proba = logreg_pipeline.predict_proba(X_test)[:,1]
print("Training Logistic Regression Done!!")

In [ ]:
print("\n=== Logistic Regression ===")
print(classification_report(y_test, logreg_pred))
print("AUC:", roc_auc_score(y_test, logreg_proba))

print("=== Logistic Regression Confusion Matrix ===")
print(confusion_matrix(y_test, logreg_pred))

In [ ]:
print("Training Logistic with Sentiment Regression...")
logreg_sentiment_pipeline.fit(X_train, y_train)
logreg_sentiment_pred = logreg_sentiment_pipeline.predict(X_test)
logreg_sentiment_proba = logreg_sentiment_pipeline.predict_proba(X_test)[:,1]
print("Training Logistic with Sentiment Regression Done!!")

In [ ]:
print("\n=== Logistic Regression with Sentiment  ===")
print(classification_report(y_test, logreg_sentiment_pred))
print("AUC:", roc_auc_score(y_test, logreg_sentiment_proba))

print("=== Logistic Regression  with Sentiment Confusion Matrix ===")
print(confusion_matrix(y_test, logreg_pred))

XgBoost

In [ ]:
# Count class distribution in TRAINING data
n_zeros = (y_train == 0).sum()
n_ones = (y_train == 1).sum()

print("Negatives:", n_zeros)
print("Positives:", n_ones)
print("Ratio (neg/pos) =", n_zeros / n_ones)

In [ ]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_lambda=2,
    reg_alpha=1,
    scale_pos_weight=1,   # <- VERY IMPORTANT
    eval_metric="logloss",
    tree_method="hist"
)

xgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", xgb_clf)
])

xgb_sentiment_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", xgb_clf)
])

In [ ]:
print("Training XGBoost...")
xgb_pipeline.fit(X_train, y_train)
xgb_pred = xgb_pipeline.predict(X_test)
xgb_proba = xgb_pipeline.predict_proba(X_test)[:,1]
print("Training XGBoost Done!!")

In [ ]:
print("\n=== XGBoost ===")
print(classification_report(y_test, xgb_pred))
print("AUC:", roc_auc_score(y_test, xgb_proba))

print("=== XGBoost Confusion Matrix ===")
print(confusion_matrix(y_test, xgb_pred))


In [ ]:
print("Training XGBoost With Sentiment Analysis...")
xgb_sentiment_pipeline.fit(X_train, y_train)
xgb_sentiment_pred = xgb_sentiment_pipeline.predict(X_test)
xgb_sentiment_proba = xgb_sentiment_pipeline.predict_proba(X_test)[:,1]
print("Training XGBoost with Sentiment Done!!")

In [ ]:
print("\n=== XGBoost with Sentiment Analysis ===")
print(classification_report(y_test, xgb_sentiment_pred))
print("AUC:", roc_auc_score(y_test, xgb_sentiment_proba))

print("=== XGBoost Confusion with Sentiment Analysis Matrix ===")
print(confusion_matrix(y_test, xgb_pred))


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "AUC": [
        roc_auc_score(y_test, logreg_sentiment_proba),
        roc_auc_score(y_test, rf_sentiment_proba),
        roc_auc_score(y_test, xgb_sentiment_proba)
    ]
})

print("\n=== Model Comparison ===")
print(comparison)

Training AUC

In [ ]:
rf_pred_train = randomforest_sentiment_pipeline.predict(X_train)
rf_proba_train = randomforest_sentiment_pipeline.predict_proba(X_train)[:,1]

print("=== Random Forest with Sentiment Analysis TRAIN AUC ===")
print(roc_auc_score(y_train, rf_proba_train))

In [ ]:
logreg_pred_train = logreg_sentiment_pipeline.predict(X_train)
logreg_proba_train = logreg_sentiment_pipeline.predict_proba(X_train)[:,1]

print("=== Logistic Regression with Sentiment Analysis TRAIN AUC ===")
print(roc_auc_score(y_train, logreg_proba_train))


In [ ]:
xgb_pred_train = xgb_sentiment_pipeline.predict(X_train)
xgb_proba_train = xgb_sentiment_pipeline.predict_proba(X_train)[:,1]

print("=== XGBoost with Sentiment TRAIN AUC ===")
print(roc_auc_score(y_train, xgb_proba_train))

5 fold validation

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler


complaints_sorted = complaint_df.sort_values('date_received').reset_index(drop=True)

X = complaints_sorted.drop(columns=[TARGET_COL])
y = complaints_sorted[TARGET_COL]

print("Total samples:", len(X))
print("Class balance:\n", y.value_counts())

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
oversampler = RandomOverSampler(random_state=42)

In [ ]:
#for cross fold validation

#Random Forest Classifier
rf_clf_cv = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt"
)

# Logistic Regression
logreg_cv = LogisticRegression(
    max_iter=500,
    C=0.5,
    solver="lbfgs",
    n_jobs=-1
)

# XGBoost
xgb_clf_cv = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_lambda=2,
    reg_alpha=1,
    scale_pos_weight=1,      # IMPORTANT: training will be balanced by oversampling
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1
)

In [ ]:
# Pipelines with preprocessing + oversampling + model
#    (using sentiment preprocessor)
rf_cv_pipeline = ImbPipeline(steps=[
    ("preprocess", preprocessor_sentiment),
    ("oversample", oversampler),
    ("model", rf_clf_cv)
])

logreg_cv_pipeline = ImbPipeline(steps=[
    ("preprocess", preprocessor_sentiment),
    ("oversample", oversampler),
    ("model", logreg_cv)
])

xgb_cv_pipeline = ImbPipeline(steps=[
    ("preprocess", preprocessor_sentiment),
    ("oversample", oversampler),
    ("model", xgb_clf_cv)
])


In [ ]:
# 6. Run 5-fold CV (AUC) for each model
def run_cv(name, pipeline):
    print(f"\n=== {name} 5-fold TimeSeries CV (AUC) ===")
    scores = cross_val_score(
        pipeline,
        X, y,
        cv=tscv,
        scoring="roc_auc",
        n_jobs=-1
    )
    print("Fold AUCs:", scores)
    print("Mean AUC:  ", scores.mean())
    print("Std AUC:   ", scores.std())
    return scores

rf_cv_scores   = run_cv("Random Forest", rf_cv_pipeline)
lg_cv_scores   = run_cv("Logistic Regression", logreg_cv_pipeline)
xgb_cv_scores  = run_cv("XGBoost", xgb_cv_pipeline)


Visualizations

In [ ]:
cv_dict = {
    "Logistic Regression": lg_cv_scores,
    "Random Forest"      : rf_cv_scores,
    "XGBoost"            : xgb_cv_scores
}

rows = []
for model_name, scores in cv_dict.items():
    for fold_idx, score in enumerate(scores, start=1):
        rows.append({
            "Model": model_name,
            "Fold": fold_idx,
            "AUC": score
        })

cv_df = pd.DataFrame(rows)
cv_df

In [ ]:
mean_auc = cv_df.groupby("Model")["AUC"].mean()

plt.figure(figsize=(6,4))
mean_auc.plot(kind="bar", color=["steelblue", "seagreen", "darkorange"])
plt.ylabel("Mean CV AUC")
plt.ylim(0.95, 1.0)
plt.title("5-fold Cross-Validation AUC by Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.style.use("ggplot")  # softer background style

fig, ax = plt.subplots(figsize=(6, 4))

# Bar plot with error bars (shows stability across folds)
ax.bar(mean_auc.index, mean_auc.values, yerr=std_auc.values, capsize=5)

# Add value labels on top of bars
for i, v in enumerate(mean_auc.values):
    ax.text(i,
            v + 0.0003,         # slightly above each bar
            f"{v:.3f}",         # e.g. 0.986
            ha="center",
            va="bottom",
            fontsize=9)

# Labels and title
ax.set_ylabel("Mean 5-fold AUC")
ax.set_xlabel("Model")
ax.set_title("5-fold Cross-Validation AUC by Model")

# Focus the y-axis on the interesting range
ax.set_ylim(0.96, 1.0)

# Remove top/right spines for a cleaner look
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
mean_auc = cv_df.groupby("Model")["AUC"].mean()
std_auc  = cv_df.groupby("Model")["AUC"].std()

plt.figure(figsize=(6,4))
plt.bar(mean_auc.index, mean_auc.values,
        yerr=std_auc.values,
        capsize=5,
        color=["steelblue", "seagreen", "darkorange"])

plt.ylabel("AUC")
plt.ylim(0.95, 1.0)
plt.title("5-fold CV AUC (Mean ± Std) by Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(x="Model", y="AUC", data=cv_df,
            palette=["steelblue", "seagreen", "darkorange"])
sns.stripplot(x="Model", y="AUC", data=cv_df,
              color="black", size=4, alpha=0.7)

plt.ylim(0.97, 1.0)
plt.title("Distribution of 5-fold CV AUC by Model")
plt.tight_layout()
plt.show()

SHAP for Random Forest

In [ ]:

# 1) Transform with the pipeline preprocessor
X_test_transformed = randomforest_pipeline.named_steps["preprocess"].transform(X_test)

# 2) Convert to dense
if sparse.issparse(X_test_transformed):
    X_dense = X_test_transformed.toarray()
else:
    X_dense = X_test_transformed

# 3) Use a subset for speed
X_sample = X_dense[:1000]
print("X_sample shape:", X_sample.shape)


In [ ]:
pre = randomforest_pipeline.named_steps["preprocess"]
text_features = pre.named_transformers_["text"].get_feature_names_out()
cat_features  = pre.named_transformers_["cat"].get_feature_names_out()
feature_names = np.concatenate([text_features, cat_features])

print("Num feature names:", len(feature_names))


In [ ]:
rf_model = randomforest_pipeline.named_steps["model"]

# Force TreeExplainer (not generic Explainer) for RF
explainer_rf = shap.TreeExplainer(
    rf_model,
    feature_perturbation="interventional",
    model_output="raw"
)

shap_values_rf = explainer_rf.shap_values(X_sample)
print("Type of shap_values_rf:", type(shap_values_rf))


In [ ]:
# Handle different SHAP versions robustly
if isinstance(shap_values_rf, list):
    # classic API: list of [class0_array, class1_array]
    shap_for_plot = shap_values_rf[1]       # class 1 = dispute
else:
    # newer API might return a 3D array: (n_samples, n_features, n_classes)
    vals = np.array(shap_values_rf)
    if vals.ndim == 3:
        shap_for_plot = vals[:, :, 1]       # class 1
    elif vals.ndim == 2:
        shap_for_plot = vals                # already 2D
    else:
        raise ValueError(f"Unexpected SHAP shape: {vals.shape}")

print("SHAP for plot shape:", shap_for_plot.shape)
print("X_sample shape:", X_sample.shape)


In [ ]:
shap.summary_plot(
    shap_for_plot,      # <-- 2D: (n_samples, n_features)
    X_sample,
    feature_names=feature_names,
    max_display=40
)


SHAP for Logistic Regression

In [ ]:
# Transform test data using the pipeline
X_test_transformed = logreg_pipeline.named_steps["preprocess"].transform(X_test)

# Convert sparse TF-IDF matrix to dense (LinearExplainer requires dense)
if sparse.issparse(X_test_transformed):
    X_dense = X_test_transformed.toarray()
else:
    X_dense = X_test_transformed

# Use a smaller subset for SHAP speed
X_sample = X_dense[:1000]
print("X_sample shape:", X_sample.shape)

In [ ]:
logreg_model = logreg_pipeline.named_steps["model"]

# SHAP for linear models (perfect for Logistic Regression)
explainer_lr = shap.LinearExplainer(
    logreg_model,
    X_sample,
    feature_names=feature_names
)

shap_values_lr = explainer_lr.shap_values(X_sample)
print("SHAP values shape:", shap_values_lr.shape)


In [ ]:
shap.summary_plot(
    shap_values_lr,
    X_sample,
    feature_names=feature_names,
    max_display=40
)


In [ ]:
shap.summary_plot(
    shap_values_lr,
    feature_names=feature_names,
    plot_type="bar",
    max_display=40
)


SHAP for XG Boost

In [ ]:
# Transform with the XGBoost pipeline preprocessor
X_test_transformed = xgb_pipeline.named_steps["preprocess"].transform(X_test)

# Convert sparse TF-IDF matrix to dense
if sparse.issparse(X_test_transformed):
    X_dense = X_test_transformed.toarray()
else:
    X_dense = X_test_transformed

# Use a subset for speed (e.g., first 1000 rows)
X_sample = X_dense[:1000]
print("X_sample shape:", X_sample.shape)

In [ ]:
pre = xgb_pipeline.named_steps["preprocess"]

text_features = pre.named_transformers_["text"].get_feature_names_out()
cat_features  = pre.named_transformers_["cat"].get_feature_names_out()

feature_names = np.concatenate([text_features, cat_features])
print("Num feature names:", len(feature_names))

In [ ]:
xgb_model = xgb_pipeline.named_steps["model"]

# TreeExplainer is the correct explainer for XGBoost
explainer_xgb = shap.TreeExplainer(
    xgb_model,
    feature_perturbation="interventional",
    model_output="raw"
)

shap_values_xgb = explainer_xgb.shap_values(X_sample)
print("Raw SHAP type:", type(shap_values_xgb))

In [ ]:
# Handle different SHAP outputs across versions
if isinstance(shap_values_xgb, list):
    # Old style: list [class0, class1]
    shap_for_plot = shap_values_xgb[1]       # class 1 = dispute
else:
    vals = np.array(shap_values_xgb)
    if vals.ndim == 3:
        # (n_samples, n_features, n_classes)
        shap_for_plot = vals[:, :, 1]        # class 1
    elif vals.ndim == 2:
        # (n_samples, n_features)
        shap_for_plot = vals
    else:
        raise ValueError(f"Unexpected SHAP shape: {vals.shape}")

print("SHAP for plot shape:", shap_for_plot.shape)
print("X_sample shape:", X_sample.shape)


In [ ]:
shap.summary_plot(
    shap_for_plot,          # (n_samples, n_features)
    X_sample,
    feature_names=feature_names,
    max_display=40
)



In [ ]:
sent_dispute = complaint_df[complaint_df['dispute_flag'] == 1]['sentiment']
sent_no_dispute = complaint_df[complaint_df['dispute_flag'] == 0]['sentiment']

# Summary statistics
print("=== Sentiment Summary by Group ===")
print("No Dispute (0):")
print(sent_no_dispute.describe())
print("\nDispute (1):")
print(sent_dispute.describe())

# Mann–Whitney U test (non-parametric)
stat, p_value = stats.mannwhitneyu(sent_no_dispute, sent_dispute, alternative='two-sided')

print("\n=== Mann–Whitney U Test ===")
print("Statistic:", stat)
print("p-value:", p_value)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.boxplot(data=complaint_df, x='dispute_flag', y='sentiment')
plt.title("Sentiment Distribution by Dispute Flag")
plt.xlabel("Dispute Flag (0 = No, 1 = Yes)")
plt.ylabel("Sentiment Score")
plt.show()

plt.figure(figsize=(10, 6))
sns.violinplot(data=complaint_df, x='dispute_flag', y='sentiment')
plt.title("Sentiment Distribution (Violin) by Dispute Flag")
plt.xlabel("Dispute Flag (0 = No, 1 = Yes)")
plt.ylabel("Sentiment Score")
plt.show()


In [ ]:
!pip install gensim
!pip install pyLDAvis
!pip install nltk


In [ ]:
import gensim
from gensim import corpora
import nltk
from nltk.corpus import stopwords
import re

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z ]', ' ', text)
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return tokens

complaint_df['tokens'] = complaint_df['consumer_complaint'].astype(str).apply(clean_text)


In [ ]:
dictionary = corpora.Dictionary(complaint_df['tokens'])
dictionary.filter_extremes(no_below=20, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in complaint_df['tokens']]


In [ ]:
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=5,
    passes=5,
    random_state=42
)

lda_model.print_topics()


In [ ]:
# Get the dominant topic for each document
all_topics = []

for bow in corpus:
    # get topic distribution for each document
    topic_probabilities = lda_model.get_document_topics(bow, minimum_probability=0.0)
    # find the topic with highest probability
    dominant_topic = sorted(topic_probabilities, key=lambda x: x[1], reverse=True)[0][0]
    all_topics.append(dominant_topic)


In [ ]:
topic_freq = [all_topics.count(i) for i in range(5)]
topic_freq


In [ ]:
topic_labels = [
    "ID Theft / Report Fix",
    "Credit Info Dispute",
    "Mortgage Issues",
    "Bank/Card Issues",
    "Payment Problems"
]

plt.figure(figsize=(10,5))
plt.bar(range(5), topic_freq)
plt.xticks(range(5), topic_labels, rotation=45, ha='right')
plt.title("Distribution of Complaint Topics (LDA)")
plt.ylabel("Number of Complaints")
plt.show()




In [ ]:
!pip install bertopic sentence-transformers


In [ ]:
from bertopic import BERTopic

docs = complaint_df["consumer_complaint"].astype(str).tolist()

topic_model = BERTopic(verbose=True)
topics, _ = topic_model.fit_transform(docs)

topic_model.get_topic_info()


In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_counts = topic_model.get_topic_info()
topic_counts.head(10)


In [ ]:
# Get topic info
topic_info = topic_model.get_topic_info()

# Keep only real topics (exclude -1 = outliers)
topic_info = topic_info[topic_info.Topic != -1]

# Get top 10 topics
top10 = topic_info.nlargest(10, 'Count')

# Add keyword string for readability
top10['Keywords'] = top10['Representation'].apply(lambda x: ", ".join(x[:6]))

top10


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(top10['Name'], top10['Count'], color='steelblue')
plt.xlabel("Number of Complaints")
plt.ylabel("Topic Name")
plt.title("Top 10 BERTopic Themes")
plt.gca().invert_yaxis()  # largest at top
plt.show()


In [ ]:
from wordcloud import WordCloud

for i, row in top10.iterrows():
    topic_id = row['Topic']
    words = dict(topic_model.get_topic(topic_id))

    wc = WordCloud(width=800, height=500, background_color='white').generate_from_frequencies(words)

    plt.figure(figsize=(6,4))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis("off")
    plt.title(f"Topic {topic_id}: {row['Name']}")
    plt.show()


In [ ]:

display(top10[['Topic', 'Count', 'Name', 'Keywords']])


In [ ]:
human_labels = {
    0: "Credit Reporting Errors",
    1: "Navient Student Loan Issues",
    2: "Bank Check/Deposit Problems",
    3: "Mortgage & Foreclosure",
    4: "Identity/Section 1681 Issues",
    5: "Escrow & Mortgage Insurance Issues",
    6: "Capital One Credit Card Issues",
    7: "TransUnion Report Disputes",
    8: "Auto Loan & Dealership Issues",
    9: "Medical Debt/Billing Problems",
    10: "Citibank Credit Card Issues"
}

top10['Human_Label'] = top10['Topic'].map(human_labels)
top10[['Topic', 'Human_Label', 'Count', 'Keywords']]


In [ ]:
complaint_df['topic'] = topics


In [ ]:
import pandas as pd

# Assuming you have:
# df["topic"] = BERTopic assigned topic
# df["dispute_flag"] = 0 or 1

topic_stats = (
    complaint_df.groupby('topic', as_index=False)   # <– ensures 'topic' is a column, not an index
        .agg(
            count=('topic', 'size'),
            dispute_rate=('dispute_flag', 'mean')
        )
)

topic_stats['dispute_rate'] = topic_stats['dispute_rate'].round(3)

print(topic_stats.head())
print(topic_stats.columns)

In [ ]:
topic_model.get_topic_info()


In [ ]:
df_topics = topic_model.get_topic_info()      # Topic, Count, Name, Representation
df_topics['Human_Label'] = df_topics['Topic'].map(human_labels)

df_topics.head()



In [ ]:
topic_stats = topic_stats.merge(
    df_topics[['Topic', 'Human_Label']],
    left_on='topic',
    right_on='Topic',
    how='left'
)

# Optionally drop the redundant 'Topic' column
topic_stats = topic_stats.drop(columns=['Topic'])

# Reorder columns nicely
topic_stats = topic_stats[['topic', 'Human_Label', 'count', 'dispute_rate']]

topic_stats.head(15)



In [ ]:
import matplotlib.pyplot as plt

top10 = plot_df.sort_values('dispute_rate', ascending=False).head(10)

plt.figure(figsize=(10,6))
plt.barh(top10['Human_Label'], top10['dispute_rate'], color='crimson')
plt.xlabel("Dispute rate")
plt.title("BERTopic Themes with Highest Dispute Rates")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# keep only rows with a label and numeric dispute rate
topic_stats['dispute_rate'] = pd.to_numeric(topic_stats['dispute_rate'], errors='coerce')

plot_df = (
    topic_stats
    .dropna(subset=['Human_Label', 'dispute_rate'])
    .sort_values('dispute_rate', ascending=False)
    .head(10)
)

plt.figure(figsize=(10,6))
plt.barh(plot_df['Human_Label'], plot_df['dispute_rate'], color='crimson')
plt.xlabel("Dispute rate")
plt.title("BERTopic Themes with Highest Dispute Rates")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
plot_df = (
    topic_stats[topic_stats['topic'] != -1]  # remove outlier cluster
      .dropna(subset=['Human_Label', 'dispute_rate'])
      .sort_values('dispute_rate', ascending=False)
      .head(10)
)


In [ ]:
plot_df.head(10)